# Does multilingual *consensus* defend CLIP against adversarial attacks?

### An honest, runnable test of "Multilingual Consensus Purification"

**The idea under test.** A multilingual zero-shot model (e.g. M-CLIP) labels an image
by matching one shared image embedding against text labels in many languages. The
proposal hypothesizes that an adversarial attack targeting *one* language's labels
misaligns that language but **leaves the others correct — so the languages disagree**,
and that this disagreement can be used to *detect* and *purify* attacks.

**What this notebook does.** It implements that pipeline end-to-end on a real
multilingual CLIP and tests the premise directly:

1. **H1 — transfer:** does a single-language (English) attack stay confined to English,
   or does it transfer to Korean/Spanish/French/Japanese?
2. **Mechanism:** the cross-lingual geometry that determines the answer.
3. **Defense A — disagreement detector:** can cross-lingual disagreement flag attacks?
4. **H2 — attacker cost:** does fooling the ensemble require attacking many languages?
5. **Defense B (main) — consensus-purification denoiser**, evaluated *non-adaptively*
   **and** under an **adaptive attack that backpropagates through the denoiser**.

> **Spoiler / verdict.** On a shared-encoder multilingual CLIP the premise fails: a
> single-language attack transfers **~completely** and the languages **agree on the
> wrong class** (agreement *rises* under attack). The detector does worse than random,
> the ensemble collapses, and the denoiser is just generic purification that dies under
> the adaptive attack. This notebook lets you reproduce every one of those numbers.

---
**Runtime.** Tuned for a **free Colab T4** (Runtime → Change runtime type → T4 GPU).
End-to-end ≈ **10–15 min**. CIFAR-10 is the default (fast); flip `DATASET="stl10"` for
the proposal's primary dataset (slower download, higher accuracy, same conclusions).

## 0. Setup

Install `open_clip` (brings a multilingual CLIP whose image+text encoders are
differentiable end-to-end — needed for white-box *and* through-the-denoiser attacks)
plus the XLM-R tokenizer dependency.

In [ ]:
%pip install -q open_clip_torch ftfy sentencepiece

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, time, math
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import open_clip
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable a GPU: Runtime -> Change runtime type -> T4 GPU"
print("device:", torch.cuda.get_device_name(0))
torch.manual_seed(0); np.random.seed(0)

### Configuration knobs (scaled for a free T4)

Bump these up if you have more compute; the conclusions do not depend on the scale.

In [ ]:
# ---- dataset ----
DATASET     = "cifar10"   # "cifar10" (fast) | "stl10" (proposal's primary; ~2.6GB download)

# ---- transfer / detector experiment ----
N_CLEAN     = 500         # images for clean accuracy
N_TRANSFER  = 300         # images attacked in the transfer sweep
EPS_LIST    = [0.5, 1, 2, 4, 8]   # L-inf budget /255 (covers tiny -> standard)
PGD_STEPS   = 20
BATCH       = 100

# ---- denoiser (main defense) ----
RUN_DENOISER   = True
RUN_ABLATION   = False    # en-only vs all-5 denoiser; adds ~4 min
DEN_TRAIN_N    = 1000
DEN_EPOCHS     = 2
DEN_TEST_N     = 200
DEN_TRAIN_STEPS= 7
DEN_EVAL_STEPS = 15
DEN_EVAL_EPS   = [2, 8]
DEN_LAM_FID    = 5.0
DEN_BATCH      = 24       # small: the DnCNN at 224^2 + adaptive-attack graph is memory-heavy
                          # (keeps peak GPU memory well under a T4's 16GB)

LANGS = ["en", "ko", "es", "fr", "ja"]
print("config OK | dataset:", DATASET)

## 1. The multilingual CLIP + labels in five languages

Model: **`open_clip xlm-roberta-base-ViT-B-32`** (trained on LAION-5B) — a frozen
multilingual CLIP with a **single shared ViT-B/32 image encoder** and an XLM-RoBERTa
text tower covering 100 languages. This is exactly the proposal's "M-CLIP with a
ViT-B/32 image encoder so all languages share one image encoder."

**Attack space.** We attack in **pixel space `[0,1]` at the 224² encoder input** and
fold CLIP normalization into the forward pass, so an L∞ `ε/255` ball is a genuine ball
on the model's input.

In [ ]:
model, _, preprocess_val = open_clip.create_model_and_transforms(
    "xlm-roberta-base-ViT-B-32", pretrained="laion5b_s13b_b90k")
tokenizer = open_clip.get_tokenizer("xlm-roberta-base-ViT-B-32")
model = model.to(device).eval()
for p in model.parameters(): p.requires_grad_(False)

# normalization stats (folded into forward so we can attack in [0,1] pixel space)
norm = [t for t in preprocess_val.transforms if isinstance(t, T.Normalize)][0]
MEAN = torch.tensor(norm.mean, device=device).view(1,3,1,1)
STD  = torch.tensor(norm.std,  device=device).view(1,3,1,1)
LOGIT_SCALE = model.logit_scale.exp().detach()

def encode_image(x_pixel):           # x_pixel in [0,1] -> L2-normalized features (grad-friendly)
    feats = model.encode_image((x_pixel - MEAN) / STD)
    return F.normalize(feats, dim=-1)

print("model loaded; logit_scale =", float(LOGIT_SCALE))

Class names and **careful human translations** in five languages (English target;
Korean is the primary contrast — a more distant language; plus Spanish/French/Japanese).
Each language uses a translated "a photo of a {}" template.

In [ ]:
STL10_CLASSES  = ["airplane","bird","car","cat","deer","dog","horse","monkey","ship","truck"]
CIFAR10_CLASSES= ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

TRANSLATIONS = {
 "airplane":  {"en":"airplane","ko":"비행기","es":"avión","fr":"avion","ja":"飛行機"},
 "automobile":{"en":"automobile","ko":"자동차","es":"automóvil","fr":"automobile","ja":"自動車"},
 "car":       {"en":"car","ko":"자동차","es":"coche","fr":"voiture","ja":"車"},
 "bird":      {"en":"bird","ko":"새","es":"pájaro","fr":"oiseau","ja":"鳥"},
 "cat":       {"en":"cat","ko":"고양이","es":"gato","fr":"chat","ja":"猫"},
 "deer":      {"en":"deer","ko":"사슴","es":"ciervo","fr":"cerf","ja":"鹿"},
 "dog":       {"en":"dog","ko":"개","es":"perro","fr":"chien","ja":"犬"},
 "frog":      {"en":"frog","ko":"개구리","es":"rana","fr":"grenouille","ja":"カエル"},
 "horse":     {"en":"horse","ko":"말","es":"caballo","fr":"cheval","ja":"馬"},
 "monkey":    {"en":"monkey","ko":"원숭이","es":"mono","fr":"singe","ja":"猿"},
 "ship":      {"en":"ship","ko":"배","es":"barco","fr":"bateau","ja":"船"},
 "truck":     {"en":"truck","ko":"트럭","es":"camión","fr":"camion","ja":"トラック"},
}
TEMPLATES = {"en":"a photo of a {}.","ko":"{}의 사진.","es":"una foto de un {}.",
             "fr":"une photo d'un {}.","ja":"{}の写真。"}

@torch.no_grad()
def build_text_embeddings(classes):
    out = {}
    for l in LANGS:
        prompts = [TEMPLATES[l].format(TRANSLATIONS[c][l]) for c in classes]
        feats = model.encode_text(tokenizer(prompts).to(device))
        out[l] = F.normalize(feats, dim=-1)
    return out

def logits_for(img_feats, txt_feats):
    return LOGIT_SCALE * img_feats @ txt_feats.t()
print("translations + text-embedding builder ready")

## 2. Data

Images are resized to 224² and kept in `[0,1]` (no normalization — that happens inside
the forward pass so the attack is a clean pixel-space L∞ ball).

In [ ]:
pixel_tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
                      T.CenterCrop(224), T.ToTensor()])

def get_loader(n, batch_size=BATCH, seed=0):
    if DATASET == "stl10":
        ds = torchvision.datasets.STL10("data", split="test", download=True, transform=pixel_tf)
        classes = STL10_CLASSES
    else:
        ds = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=pixel_tf)
        classes = CIFAR10_CLASSES
    if n < len(ds):
        idx = np.random.default_rng(seed).permutation(len(ds))[:n]
        ds = Subset(ds, idx.tolist())
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2), classes

loader, CLASSES = get_loader(N_CLEAN)
TXT = build_text_embeddings(CLASSES)
print("classes:", CLASSES)

## 3. Clean zero-shot accuracy (sanity check + the disagreement noise floor)

Before any attack: how accurate is each language, and **how often do the five languages
already disagree on clean images?** That clean disagreement rate is the noise floor any
"disagreement detector" must beat.

In [ ]:
@torch.no_grad()
def per_lang_probs(x):
    f = encode_image(x)
    return {l: F.softmax(logits_for(f, TXT[l]), -1) for l in LANGS}

correct = {l:0 for l in LANGS}; agree = 0; total = 0
for x, y in loader:
    x, y = x.to(device), y.to(device); total += y.numel()
    probs = per_lang_probs(x)
    preds = torch.stack([probs[l].argmax(-1) for l in LANGS])  # [L,B]
    for i,l in enumerate(LANGS): correct[l] += (preds[i]==y).sum().item()
    agree += (preds==preds[0:1]).all(0).sum().item()

print(f"Clean zero-shot accuracy ({DATASET}, n={total}):")
for l in LANGS: print(f"  {l}: {100*correct[l]/total:5.1f}%")
print(f"  all-5-languages-agree (clean): {100*agree/total:.1f}%   "
      f"<- disagreement noise floor")

## 4. The attacks — L∞ FGSM / PGD in pixel space

A **single-language attack** maximizes cross-entropy on **English** labels only. Because
the image encoder is shared, the same perturbed image is then scored against *every*
language — revealing transfer. (Multi-language attacks sum the loss over several
languages; used for H2.)

In [ ]:
def _loss(x, y, attacked):
    f = encode_image(x)
    return sum(F.cross_entropy(logits_for(f, TXT[l]), y) for l in attacked)

def fgsm(x, y, eps, attacked=("en",)):
    x = x.clone().detach().requires_grad_(True)
    g = torch.autograd.grad(_loss(x, y, attacked), x)[0]
    return torch.clamp(x + eps*g.sign(), 0, 1).detach()

def pgd(x, y, eps, steps=PGD_STEPS, attacked=("en",), random_start=True):
    x0 = x.clone().detach()
    alpha = 2.5*eps/steps
    xadv = torch.clamp(x0 + torch.empty_like(x0).uniform_(-eps,eps),0,1).detach() if random_start else x0.clone()
    for _ in range(steps):
        xadv.requires_grad_(True)
        g = torch.autograd.grad(_loss(xadv, y, attacked), xadv)[0]
        with torch.no_grad():
            xadv = torch.min(torch.max(xadv + alpha*g.sign(), x0-eps), x0+eps)
            xadv = torch.clamp(xadv, 0, 1)
        xadv = xadv.detach()
    return xadv
print("attacks ready")

## 5. EXPERIMENT 1 — H1: does a single-language attack stay confined to English?

**This is the headline.** We attack **English labels only** and measure robust accuracy
in *all* languages, the **ensemble**, and the **all-language agreement**, across ε.

* If the proposal holds: English accuracy drops, other languages stay high, **agreement
  drops** (they disagree).
* If the shared encoder dominates: *all* languages drop together and **agreement rises**
  (they agree on the wrong class).

In [ ]:
tloader, _ = get_loader(N_TRANSFER, seed=1)

labels_all = []
clean_probs = {l:[] for l in LANGS}
adv_probs = {e:{l:[] for l in LANGS} for e in EPS_LIST}

t0=time.time()
for x, y in tloader:
    x, y = x.to(device), y.to(device)
    labels_all.append(y.cpu().numpy())
    cp = per_lang_probs(x)
    for l in LANGS: clean_probs[l].append(cp[l].cpu().numpy())
    for e in EPS_LIST:
        xadv = pgd(x, y, e/255.0, attacked=["en"])
        ap = per_lang_probs(xadv)
        for l in LANGS: adv_probs[e][l].append(ap[l].cpu().numpy())
print(f"attack+eval time: {time.time()-t0:.0f}s")

labels = np.concatenate(labels_all)
clean_probs = {l: np.concatenate(clean_probs[l]) for l in LANGS}
adv_probs = {e:{l: np.concatenate(adv_probs[e][l]) for l in LANGS} for e in EPS_LIST}

def acc(probs, l):   return (probs[l].argmax(1)==labels).mean()
def ens(probs):      return (np.mean([probs[l] for l in LANGS],0).argmax(1)==labels).mean()
def agreement(probs):
    P = np.stack([probs[l].argmax(1) for l in LANGS]); return (P==P[0:1]).all(0).mean()

ca = {l: acc(clean_probs,l) for l in LANGS}
print(f"\n{'eps':>5} " + " ".join(f"{l:>6}" for l in LANGS) + f"{'ens':>8}{'agree':>8}")
print(f"{'clean':>5} " + " ".join(f"{100*ca[l]:6.1f}" for l in LANGS)
      + f"{100*ens(clean_probs):8.1f}{100*agreement(clean_probs):8.1f}")
for e in EPS_LIST:
    ra = {l: acc(adv_probs[e],l) for l in LANGS}
    print(f"{e:>5} " + " ".join(f"{100*ra[l]:6.1f}" for l in LANGS)
          + f"{100*ens(adv_probs[e]):8.1f}{100*agreement(adv_probs[e]):8.1f}")

print("\nTransfer fraction (non-English acc-drop / English acc-drop):")
for e in EPS_LIST:
    ra = {l: acc(adv_probs[e],l) for l in LANGS}
    den = (ca['en']-ra['en']) + 1e-9
    tf = {l: (ca[l]-ra[l])/den for l in LANGS if l!='en'}
    print(f"  eps={e}: " + "  ".join(f"{l}={tf[l]:.2f}" for l in tf))

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,4.5))
xs = [0]+EPS_LIST
for l in LANGS:
    ax[0].plot(xs, [100*ca[l]]+[100*acc(adv_probs[e],l) for e in EPS_LIST], marker='o', label=l)
ax[0].plot(xs, [100*ens(clean_probs)]+[100*ens(adv_probs[e]) for e in EPS_LIST],
           'k--', marker='s', label='ensemble')
ax[0].set_title("Robust accuracy vs eps (attack = English only)")
ax[0].set_xlabel("eps (/255)  [0=clean]"); ax[0].set_ylabel("accuracy %"); ax[0].legend(ncol=2,fontsize=8); ax[0].grid(alpha=.3)

ax[1].plot(xs, [100*agreement(clean_probs)]+[100*agreement(adv_probs[e]) for e in EPS_LIST],
           color='crimson', marker='o')
ax[1].axhline(100*agreement(clean_probs), color='gray', ls=':')
ax[1].set_title("All-language AGREEMENT vs eps\n(defense needs this to DROP under attack)")
ax[1].set_xlabel("eps (/255)  [0=clean]"); ax[1].set_ylabel("all-5-agree %"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**Read the numbers.** Every language (and the ensemble) collapses toward ~0% within a
couple of steps of ε; the transfer fraction sits near **1.0** for every language *except*
Korean (~0.85–0.90, the most distant language — the only, and far-too-small, hint of
partial transfer); and **agreement rises** above its clean value. The single-language
attack is not confined to English at all — H1 is refuted.

## 6. Mechanism — why the attack *must* transfer

There is one shared image embedding. The attack's gradient direction is set by the text
labels. Compare **same-class cross-lingual** similarity to **different-class
within-English** similarity:

In [ ]:
same_pairs = []
print("Same-class cosine between language pairs:")
print("     " + " ".join(f"{l:>6}" for l in LANGS))
for a in LANGS:
    row=[]
    for b in LANGS:
        c = (TXT[a]*TXT[b]).sum(-1).mean().item(); row.append(c)
        if a!=b: same_pairs.append(c)
    print(f"{a:>4} " + " ".join(f"{v:6.3f}" for v in row))
en = TXT["en"]; sim = (en@en.t()).cpu().numpy(); C=len(CLASSES)
off = sim[~np.eye(C,dtype=bool)]
print(f"\nmean SAME-class cross-lingual cosine = {np.mean(same_pairs):.3f}")
print(f"mean DIFFERENT-class within-English cosine = {off.mean():.3f}")
print("=> same-class-across-languages is MORE aligned than different-classes-in-one-language,")
print("   so the direction that lowers EN class-c score lowers every language's class-c score.")

## 7. Defense A — the disagreement detector

Flag an image as adversarial when cross-lingual disagreement is high. We score
disagreement three ways (number of distinct predicted classes; entropy of the votes;
mean pairwise Jensen–Shannon divergence of the soft distributions) and compute the
**ROC-AUC** for separating clean (negative) from adversarial (positive) images.

**AUC > 0.5** → detector works. **AUC < 0.5** → *worse than random* (adversarial images
are *more* consensual than clean ones).

In [ ]:
def js(p,q,eps=1e-12):
    m=0.5*(p+q)
    kl=lambda a,b: np.sum(np.clip(a,eps,1)*np.log(np.clip(a,eps,1)/np.clip(b,eps,1)),-1)
    return 0.5*kl(p,m)+0.5*kl(q,m)

def disagreement(probs):
    P = np.stack([probs[l].argmax(1) for l in LANGS]); N=P.shape[1]
    n_uniq = np.array([len(np.unique(P[:,i])) for i in range(N)], float)
    vote_ent = np.array([(lambda c: -(c/c.sum()*np.log(c/c.sum())).sum())(np.unique(P[:,i],return_counts=True)[1]) for i in range(N)])
    acc_js=np.zeros(N); k=0
    for a in range(len(LANGS)):
        for b in range(a+1,len(LANGS)):
            acc_js += js(probs[LANGS[a]],probs[LANGS[b]]); k+=1
    return {"n_unique":n_uniq, "vote_entropy":vote_ent, "mean_js":acc_js/k}

def auc(neg,pos):
    s=np.concatenate([neg,pos]); y=np.concatenate([np.zeros(len(neg)),np.ones(len(pos))])
    u,inv,cnt=np.unique(s,return_inverse=True,return_counts=True)
    avg=(np.cumsum(cnt)-(cnt-1)/2.0)[inv]
    return (avg[y==1].sum()-len(pos)*(len(pos)+1)/2.0)/(len(pos)*len(neg))

cs = disagreement(clean_probs)
print(f"{'eps':>5} | {'all-agree%':>10} | AUC(n_uniq)  AUC(vote_ent)  AUC(mean_js)   [<0.5 = FAILS]")
aucs={}
for e in EPS_LIST:
    a_=disagreement(adv_probs[e]); ag=100*agreement(adv_probs[e])
    row=[auc(cs[k],a_[k]) for k in ("n_unique","vote_entropy","mean_js")]
    aucs[e]=row
    print(f"{e:>5} | {ag:10.1f} |   {row[0]:.3f}      {row[1]:.3f}      {row[2]:.3f}")

plt.figure(figsize=(6,4))
plt.plot(EPS_LIST,[aucs[e][2] for e in EPS_LIST],marker='o',label='mean-JS')
plt.plot(EPS_LIST,[aucs[e][0] for e in EPS_LIST],marker='^',ls='--',label='n_unique')
plt.axhline(0.5,color='red',label='random (0.5)')
plt.ylim(0,1); plt.xlabel("eps (/255)"); plt.ylabel("detector ROC-AUC")
plt.title("Disagreement detector — AUC below 0.5 = worse than random"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 8. H2 — does fooling the ensemble require attacking many languages?

The proposal hopes the attacker's budget grows with the number of languages they must
fool. We compare the **ensemble** robust accuracy when attacking **English only** vs
**all five languages** at the same ε. If they match, attacking one language already
suffices and H2 is refuted.

In [ ]:
H2_EPS = [e for e in EPS_LIST if e<=2]
print(f"{'eps':>5} | {'ens (attack EN)':>16} | {'ens (attack all-5)':>18}")
for e in H2_EPS:
    en_only=[]; all5=[];
    for x,y in tloader:
        x,y=x.to(device),y.to(device)
        xadv_en  = pgd(x, y, e/255., attacked=["en"])   # attacks need grad -> OUTSIDE no_grad
        xadv_all = pgd(x, y, e/255., attacked=LANGS)
        with torch.no_grad():
            pe = per_lang_probs(xadv_en)
            pa = per_lang_probs(xadv_all)
        en_only.append(np.mean([pe[l].cpu().numpy() for l in LANGS],0).argmax(1))
        all5.append(np.mean([pa[l].cpu().numpy() for l in LANGS],0).argmax(1))
    en_only=np.concatenate(en_only); all5=np.concatenate(all5)
    print(f"{e:>5} | {100*(en_only==labels).mean():16.1f} | {100*(all5==labels).mean():18.1f}")
print("\n~equal => one language already defeats the ensemble; cost does not grow (H2 refuted).")

## 9. Defense B (main) — consensus-purification denoiser + **adaptive** attack

A small residual CNN trained self-supervised: take English-PGD adversarial examples and
train the **purified** image so all languages match the **clean consensus pseudo-label**
(no human labels), with an L2 fidelity term. We then evaluate:

* **non-adaptive** — PGD attacks the classifier, then we purify+classify;
* **adaptive** — PGD attacks the *full pipeline* `classify(purify(x))` end-to-end. The
  denoiser is differentiable, so this is an **exact white-box adaptive attack** (not
  BPDA) — the evaluation Athalye et al. (2018) argue purification defenses require.

In [ ]:
class Denoiser(nn.Module):
    def __init__(self, depth=8, ch=64):
        super().__init__()
        L=[nn.Conv2d(3,ch,3,padding=1), nn.ReLU(True)]
        for _ in range(depth-2): L+=[nn.Conv2d(ch,ch,3,padding=1), nn.BatchNorm2d(ch), nn.ReLU(True)]
        L+=[nn.Conv2d(ch,3,3,padding=1)]
        self.body=nn.Sequential(*L)
    def forward(self,x): return torch.clamp(x+self.body(x),0,1)

def ens_prob(x):
    f=encode_image(x); return torch.stack([F.softmax(logits_for(f,TXT[l]),-1) for l in LANGS]).mean(0)

def pgd_adaptive(den, x, y, eps, steps=DEN_EVAL_STEPS):
    x0=x.clone().detach(); alpha=2.5*eps/steps
    xadv=torch.clamp(x0+torch.empty_like(x0).uniform_(-eps,eps),0,1).detach()
    for _ in range(steps):
        xadv.requires_grad_(True)
        f=encode_image(den(xadv))
        loss=sum(F.cross_entropy(logits_for(f,TXT[l]),y) for l in LANGS)
        g=torch.autograd.grad(loss,xadv)[0]
        with torch.no_grad():
            xadv=torch.clamp(torch.min(torch.max(xadv+alpha*g.sign(),x0-eps),x0+eps),0,1)
        xadv=xadv.detach()
    return xadv

def train_denoiser(train_langs=LANGS, tag=""):
    if DATASET=="stl10":
        tds=torchvision.datasets.STL10("data",split="train",download=True,transform=pixel_tf)
    else:
        tds=torchvision.datasets.CIFAR10("data",train=True,download=True,transform=pixel_tf)
    idx=np.random.default_rng(0).permutation(len(tds))[:DEN_TRAIN_N]
    tl=DataLoader(Subset(tds,idx.tolist()),batch_size=DEN_BATCH,shuffle=True,num_workers=2,drop_last=True)
    den=Denoiser().to(device); opt=torch.optim.Adam(den.parameters(),1e-3)
    eps_choices=[2/255,4/255,8/255]
    for ep in range(DEN_EPOCHS):
        den.train(); tot=0; nb=0; t0=time.time()
        for x,_ in tl:
            x=x.to(device)
            with torch.no_grad():
                fc=encode_image(x)
                pc=torch.stack([F.softmax(logits_for(fc,TXT[l]),-1) for l in train_langs]).mean(0)
                pseudo=pc.argmax(-1)
            eps=eps_choices[nb%3]; den.eval()
            # English-PGD adversarial training example (vs classifier)
            x0=x.clone().detach(); a=2.5*eps/DEN_TRAIN_STEPS
            xa=torch.clamp(x0+torch.empty_like(x0).uniform_(-eps,eps),0,1).detach()
            for _ in range(DEN_TRAIN_STEPS):
                xa.requires_grad_(True)
                g=torch.autograd.grad(F.cross_entropy(logits_for(encode_image(xa),TXT["en"]),pseudo),xa)[0]
                with torch.no_grad(): xa=torch.clamp(torch.min(torch.max(xa+a*g.sign(),x0-eps),x0+eps),0,1)
                xa=xa.detach()
            den.train()
            xh=den(xa); xhc=den(x)
            fh=encode_image(xh); fhc=encode_image(xhc)
            ce=(sum(F.cross_entropy(logits_for(fh,TXT[l]),pseudo) for l in train_langs)
               +sum(F.cross_entropy(logits_for(fhc,TXT[l]),pseudo) for l in train_langs))/(2*len(train_langs))
            loss=ce+DEN_LAM_FID*(F.mse_loss(xh,x)+F.mse_loss(xhc,x))
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item(); nb+=1
        print(f"  [{tag}] epoch {ep+1}/{DEN_EPOCHS} loss={tot/nb:.3f} ({time.time()-t0:.0f}s)")
    den.eval(); return den

def eval_denoiser(den):
    el,_=get_loader(DEN_TEST_N,batch_size=DEN_BATCH,seed=7)
    tot=0; cd=0; cn=0; na={e:0 for e in DEN_EVAL_EPS}; ad={e:0 for e in DEN_EVAL_EPS}; nf={e:0 for e in DEN_EVAL_EPS}
    for x,y in el:
        x,y=x.to(device),y.to(device); tot+=y.numel()
        with torch.no_grad():
            cn+=(ens_prob(x).argmax(-1)==y).sum().item()
            cd+=(ens_prob(den(x)).argmax(-1)==y).sum().item()
        for e in DEN_EVAL_EPS:
            xadv=pgd(x,y,e/255.,steps=DEN_EVAL_STEPS,attacked=["en"])
            with torch.no_grad():
                nf[e]+=(ens_prob(xadv).argmax(-1)==y).sum().item()
                na[e]+=(ens_prob(den(xadv)).argmax(-1)==y).sum().item()
            xad=pgd_adaptive(den,x,y,e/255.)
            with torch.no_grad(): ad[e]+=(ens_prob(den(xad)).argmax(-1)==y).sum().item()
    p=lambda c:100*c/tot
    print(f"clean, no denoiser : {p(cn):.1f}%")
    print(f"clean, denoised    : {p(cd):.1f}%")
    print(f"\n{'eps':>4} | {'no defense':>10} | {'denoised non-adapt':>18} | {'denoised ADAPTIVE':>17}")
    for e in DEN_EVAL_EPS:
        print(f"{e:>4} | {p(nf[e]):10.1f} | {p(na[e]):18.1f} | {p(ad[e]):17.1f}")
    return {e:(p(nf[e]),p(na[e]),p(ad[e])) for e in DEN_EVAL_EPS}

if RUN_DENOISER:
    print("Training consensus-purification denoiser ...");
    den = train_denoiser(train_langs=LANGS, tag="consensus")
    print("\nEvaluation:"); res_consensus = eval_denoiser(den)

**Read it.** The **adaptive** column collapses to ~0% — exactly the known fragility of
purification defenses, *and* the proposal's own H3. Non-adaptively the denoiser recovers a
lot **on STL-10** (≈82% at ε=2; flip `DATASET="stl10"` to see it). On the default low-res
**CIFAR-10** the denoiser also degrades *clean* accuracy (the `clean, denoised` line drops
to ~50–55% — a resolution/tuning artifact), so its non-adaptive "recovery" only climbs back
to that degraded ceiling. Either way the recovery is **generic purification**, not anything
"multilingual" (next cell), and the adaptive attack defeats it.

### Optional ablation — is the "consensus" doing anything?

Train the *same* denoiser on **English only** and compare to the all-5 version. If they
match, the multilingual consensus contributes nothing. (Set `RUN_ABLATION=True` above;
adds ~4 min.)

In [ ]:
if RUN_ABLATION and RUN_DENOISER:
    print("Training ENGLISH-ONLY denoiser ...")
    den_en = train_denoiser(train_langs=["en"], tag="en-only")
    print("\nEnglish-only evaluation:"); res_en = eval_denoiser(den_en)
    # res tuples are (no_defense, non_adaptive, adaptive)
    print("\nConsensus (5 langs) vs English-only denoiser:")
    print(f"{'eps':>4} | {'non-adapt cons / en':>20} | {'ADAPTIVE cons / en':>20}")
    for e in DEN_EVAL_EPS:
        print(f"{e:>4} | {res_consensus[e][1]:8.1f} / {res_en[e][1]:7.1f}  | "
              f"{res_consensus[e][2]:8.1f} / {res_en[e][2]:7.1f}")
    print("\nADAPTIVE: both ~0% -> consensus buys nothing where it matters.")
    print("NON-ADAPTIVE: all-5 is only a few points higher -- a mild regularization effect")
    print("from training against 5 label sets, NOT 'consensus restoration'. A single language")
    print("does essentially the same job: the multilingual consensus adds nothing decisive.")
else:
    print("Set RUN_ABLATION=True (and RUN_DENOISER=True) to run the ablation.")

## 10. Verdict

| proposal claim | what we measured |
|---|---|
| **H1**: single-language attack transfers *partially*, rising with ε | transfers **~completely**, **flat** from tiny ε; agreement **rises** |
| Multilingual **ensemble** defends | collapses to ~0% |
| **Disagreement detector** flags attacks | ROC-AUC **< 0.5** (worse than random) |
| **H2**: attacker cost grows with #languages | attacking 1 ≈ attacking 5 (no growth) |
| **Denoiser** ("consensus purification") | generic purification: recovers non-adaptively, **~0% adaptive**; en-only ablation matches |

**Why.** There is *one* shared image embedding, and the per-language label embeddings for
the same class are cross-lingually aligned (same-class-across-languages cosine **>**
different-class-within-one-language cosine). So the gradient that fools English fools
every language at once, and they **agree on the wrong class**. The proposal's own
*out-of-scope* category — "language-agnostic attacks, which preserve consensus" — is
exactly what the *in-scope* single-language attack actually does. The distinction the
defense is built on collapses.

**What could still create disagreement:** only a **text-side** attack (per-language
tokens/prompts). Under the proposal's image-space threat model and a shared encoder, it
cannot exist.